In [3]:
%pip install scikit-learn
import os
import json
import boto3
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

print("Libraries imported successfully!")

  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Libraries imported successfully!


In [6]:
import boto3

session = boto3.Session()

region = session.region_name

s3 = session.client("s3")

bucket_name = "sagemaker-houseprice-aastmt"

print("AWS Region:", region)
print("S3 Bucket:", bucket_name)

AWS Region: eu-north-1
S3 Bucket: sagemaker-houseprice-aastmt


In [7]:
PROJECT_DIR = "house-price-prediction"

folders = [
    "dataset",
    "model",
    "predictions",
    "lambda"
]

# Create local folders
for folder in folders:

    path = os.path.join(PROJECT_DIR, folder)

    os.makedirs(path, exist_ok=True)

# Create folders inside S3
for folder in folders:

    s3.put_object(
        Bucket=bucket_name,
        Key=f"{folder}/",
        Body=b""
    )

print("All project folders created successfully!")

All project folders created successfully!


In [8]:
response = s3.list_objects_v2(
    Bucket=bucket_name,
    Delimiter="/"
)

for item in response.get("CommonPrefixes", []):
    print(item["Prefix"])

dataset/
lambda/
model/
predictions/


In [9]:
np.random.seed(42)

number_of_houses = 500

area = np.random.uniform(
    60, 300, number_of_houses
)

bedrooms = np.random.randint(
    1, 7, number_of_houses
)

bathrooms = np.random.randint(
    1, 5, number_of_houses
)

floors = np.random.randint(
    1, 4, number_of_houses
)

age = np.random.randint(
    0, 41, number_of_houses
)

price = (
    250000
    + area * 8500
    + bedrooms * 90000
    + bathrooms * 65000
    + floors * 45000
    - age * 12000
    + np.random.normal(
        0, 150000, number_of_houses
    )
)

price = np.maximum(price, 100000)

df = pd.DataFrame({
    "area": area,
    "bedrooms": bedrooms,
    "bathrooms": bathrooms,
    "floors": floors,
    "age": age,
    "price": price
})

df = df.round(2)

df.head()

,area,bedrooms,bathrooms,floors,age,price
0,149.89,6,3,3,19,2161544.52
1,288.17,1,2,2,25,2748339.89
2,235.68,1,3,1,28,2366226.10
3,203.68,1,1,1,0,2365426.15
4,97.44,5,4,1,34,1406825.94


In [10]:
dataset_path = os.path.join(
    PROJECT_DIR,
    "dataset",
    "house_data.csv"
)

df.to_csv(
    dataset_path,
    index=False
)

print("Dataset saved successfully!")
print("File:", dataset_path)

Dataset saved successfully!
File: house-price-prediction/dataset/house_data.csv


In [11]:
s3.upload_file(
    dataset_path,
    bucket_name,
    "dataset/house_data.csv"
)

print("Dataset uploaded to S3 successfully!")

Dataset uploaded to S3 successfully!


In [12]:
response = s3.get_object(
    Bucket=bucket_name,
    Key="dataset/house_data.csv"
)

df = pd.read_csv(response["Body"])

print("Dataset loaded from S3!")
print("Dataset Shape:", df.shape)

df.head()

Dataset loaded from S3!
Dataset Shape: (500, 6)


,area,bedrooms,bathrooms,floors,age,price
0,149.89,6,3,3,19,2161544.52
1,288.17,1,2,2,25,2748339.89
2,235.68,1,3,1,28,2366226.10
3,203.68,1,1,1,0,2365426.15
4,97.44,5,4,1,34,1406825.94


In [13]:
features = [
    "area",
    "bedrooms",
    "bathrooms",
    "floors",
    "age"
]

X = df[features]

y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 400
Testing records: 100


In [14]:
model = LinearRegression()

model.fit(
    X_train,
    y_train
)

print("Model training completed!")

Model training completed!


In [15]:
predictions = model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    predictions
)

r2 = r2_score(
    y_test,
    predictions
)

print("Mean Absolute Error:", round(mae, 2))
print("R2 Score:", round(r2, 4))

Mean Absolute Error: 129028.84
R2 Score: 0.9287


In [16]:
model_data = {
    "features": features,
    "intercept": float(model.intercept_),
    "coefficients": [
        float(value)
        for value in model.coef_
    ],
    "mae": float(mae),
    "r2_score": float(r2)
}

model_path = os.path.join(
    PROJECT_DIR,
    "model",
    "model.json"
)

with open(model_path, "w") as file:
    json.dump(
        model_data,
        file,
        indent=4
    )

print("Model parameters saved successfully!")

Model parameters saved successfully!


In [17]:
s3.upload_file(
    model_path,
    bucket_name,
    "model/model.json"
)

print("Model uploaded to S3 successfully!")

Model uploaded to S3 successfully!


In [18]:
results = X_test.copy()

results["actual_price"] = y_test.values

results["predicted_price"] = predictions

prediction_path = os.path.join(
    PROJECT_DIR,
    "predictions",
    "test_predictions.csv"
)

results.to_csv(
    prediction_path,
    index=False
)

s3.upload_file(
    prediction_path,
    bucket_name,
    "predictions/test_predictions.csv"
)

print("Predictions uploaded successfully!")

Predictions uploaded successfully!
